<a href="https://colab.research.google.com/github/inesnair/birdsfinder/blob/bloc03-mode-chat/notebooks/03_mode_chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐦 BirdsFinder — Notebook 03 : le mode chat (Bloc 03, ML classique)

**Objectif** : identifier un oiseau à partir d'une description (12 questions), avec un modèle de machine learning classique, puis le brancher sur la route `POST /chat` de l'API.

| Partie | Où | Durée | Ce que ça produit |
|---|---|---|---|
| **⓪ Pousser le kit** | ton PC | 5 min | le code du Bloc 03 sur GitHub (branche `bloc03-mode-chat`) |
| **0. Préparation** | Colab | 2 min | Drive monté, code récupéré |
| **C. Test humain** | Colab | ~1 h 30 | 80 photos décrites → `descriptions_test.csv` (le vrai jeu de test) |
| **A. Modèle** | Colab | ~45 min | 5 algorithmes comparés, choix argumenté, `chat_antilles.joblib` |
| **B. API locale** | ton PC | ~30 min | `/chat` qui répond, démo Bruno, tests |

👉 **Ordre conseillé ce soir : ⓪ → 0 → C → A → B.** Le test humain passe en premier parce que la partie A l'utilise pour la note finale.

Compétences visées : **C1** choisir l'algorithme en comparant · **C2** prétraiter les données · **C3** entraîner en optimisant une loss, avec des indicateurs de succès définis.

⚙️ Pas besoin de GPU pour ce notebook (Exécution > Modifier le type d'exécution > CPU).

---
# ⓪ Avant d'ouvrir Colab : pousser le kit sur GitHub (sur ton PC)

Colab récupère le code depuis GitHub : il faut donc d'abord y envoyer les fichiers du kit.

1. Dans le terminal de VS Code, dans le dossier `birdsfinder`, repars de la dernière version de `main` :
```
git checkout main
git pull
```
2. Dézippe `kit_bloc03.zip` **dans le dossier `birdsfinder`** (accepte de remplacer `app/main.py`, `app/config.py`, `README.md`, `requirements.txt`, `requirements-ci.txt`, `tests/test_api.py`).
3. Crée la branche et envoie uniquement les fichiers du kit :
```
git checkout -b bloc03-mode-chat
git add app src tests bruno data/bloc03 notebooks/03_mode_chat.ipynb README.md requirements.txt requirements-ci.txt
git commit -m "Bloc 03 : mode chat (générateur, modèles, route /chat, tests, Bruno, notebook 03)"
git push -u origin bloc03-mode-chat
```
4. Sur GitHub, l'onglet **Actions** lance la CI : les tests (dont les 8 nouveaux du chat) doivent passer au vert.
5. Ouvre ce notebook dans Colab depuis GitHub (bouton « Open in Colab » en haut, ou Fichier > Ouvrir > GitHub > `inesnair/birdsfinder`, branche `bloc03-mode-chat`).

---
# 0. Préparation
Connexion au Drive (pour garder les fichiers), récupération du code de la branche `bloc03-mode-chat`, installation des bibliothèques.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/birdsfinder
!git clone -b bloc03-mode-chat https://github.com/inesnair/birdsfinder.git /content/birdsfinder
%cd /content/birdsfinder
!pip -q install -r requirements-train.txt

In [ ]:
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/birdsfinder")
PHOTOS = DRIVE / "data_antilles_10" / "images_propres"   # les 948 photos nettoyées (notebook 02)
BLOC03 = DRIVE / "bloc03"                                 # tout ce que produit ce notebook
TEST_HUMAIN = BLOC03 / "test_humain"
BLOC03.mkdir(parents=True, exist_ok=True)

print("Photos nettoyées trouvées :", PHOTOS.exists())
print("Dossier de sortie :", BLOC03)

---
# C. Le test humain (le vrai jeu de test)

**Pourquoi ?** Le modèle va apprendre sur des observations *simulées*. Pour savoir s'il marche vraiment, il faut le noter sur des descriptions faites par un humain, à partir de vraies photos.

**Le risque : le biais de connaissance.** Je connais les espèces et j'ai rempli les fiches : je pourrais décrire *ce que je sais* plutôt que *ce que je vois*. Les garde-fous :
1. photos **tirées au hasard** dans le jeu de test du modèle photo, **mélangées** et **renommées** (`photo_01.jpg`…) ;
2. la correspondance photo → espèce est rangée dans un fichier à part, **que je n'ouvre pas** ;
3. **règle d'or : je ne décris que ce qui est visible**. Ce qui ne se voit pas = « je ne sais pas » (c'est le choix par défaut) ;
4. environ **20 secondes par photo**.

À dire au jury : *« Test réalisé par la conceptrice, avec ces garde-fous. Le score est probablement un peu optimiste. »*

## C1. Tirage et anonymisation des 80 photos

On refait **exactement** le découpage de `src/train_cnn.py` (graine 42 : 70 % train / 15 % validation / 15 % test), puis on tire **8 photos par espèce dans le test**. Ces photos n'ont jamais servi à entraîner le modèle photo.

⚠️ La cellule ne refait rien si le tirage existe déjà (pour ne pas mélanger une 2e fois en cours de route).

In [ ]:
import shutil
import pandas as pd
from torchvision import datasets
from sklearn.model_selection import train_test_split

PHOTOS_ANONYMES = TEST_HUMAIN / "photos"
CORRESPONDANCE = TEST_HUMAIN / "NE_PAS_OUVRIR_correspondance.csv"

if CORRESPONDANCE.exists():
    print("Tirage déjà fait :", len(pd.read_csv(CORRESPONDANCE)), "photos. On ne refait rien.")
else:
    # 1) Même lecture et même découpage que train_cnn.py
    probe = datasets.ImageFolder(PHOTOS)
    classes, targets = probe.classes, probe.targets
    idx = list(range(len(targets)))
    train_idx, rest_idx = train_test_split(idx, test_size=0.30, stratify=targets, random_state=42)
    rest_targets = [targets[i] for i in rest_idx]
    val_idx, test_idx = train_test_split(rest_idx, test_size=0.50, stratify=rest_targets, random_state=42)
    print(f"{len(targets)} photos | train={len(train_idx)} | val={len(val_idx)} | test={len(test_idx)}")
    # contrôle : le journal du 20/11 indique 948 photos et 143 en test

    # 2) 8 photos par espèce, tirées au hasard dans le test, puis tout mélangé
    test = pd.DataFrame({"chemin": [probe.samples[i][0] for i in test_idx],
                         "espece_id": [classes[targets[i]] for i in test_idx]})
    tirage = test.groupby("espece_id").sample(n=8, random_state=42)
    tirage = tirage.sample(frac=1, random_state=2026).reset_index(drop=True)
    tirage["photo"] = [f"photo_{i + 1:02d}.jpg" for i in range(len(tirage))]

    # 3) Copie anonymisée + correspondance rangée à part
    PHOTOS_ANONYMES.mkdir(parents=True, exist_ok=True)
    for r in tirage.itertuples():
        shutil.copy(r.chemin, PHOTOS_ANONYMES / r.photo)
    tirage[["photo", "espece_id", "chemin"]].to_csv(CORRESPONDANCE, index=False)
    print(len(tirage), "photos anonymisées dans", PHOTOS_ANONYMES)

## C2. Décrire les 80 photos

Un petit formulaire : une photo, 12 listes déroulantes, un bouton. Chaque réponse est enregistrée **tout de suite** dans `descriptions_test.csv` sur le Drive : on peut s'arrêter et reprendre plus tard (relancer la cellule reprend à la première photo non décrite).

- Toutes les listes commencent sur **« je ne sais pas »** : je ne change que ce que **je vois**.
- **Taille** : estime-la avec le décor (branche, fil, fleur…). Si rien ne permet de juger : « je ne sais pas ».
- **Que faisait-il ? / Où ?** : seulement si la photo le montre.

🛟 *Plan B si le formulaire ne s'affiche pas* : ouvre les photos dans le Drive et remplis un tableur avec les colonnes `photo` + les 12 attributs (mêmes valeurs), à enregistrer sous `descriptions_test.csv` dans le même dossier.

In [ ]:
import csv
import ipywidgets as w
from IPython.display import display
from src.chat_attributes import ATTRIBUTS, QUESTIONS, VALEURS, JE_NE_SAIS_PAS

DESCRIPTIONS = TEST_HUMAIN / "descriptions_test.csv"
photos = sorted(p.name for p in PHOTOS_ANONYMES.glob("photo_*.jpg"))
deja = set(pd.read_csv(DESCRIPTIONS)["photo"]) if DESCRIPTIONS.exists() else set()
a_faire = [p for p in photos if p not in deja]
print(f"{len(deja)} photos déjà décrites, {len(a_faire)} à faire.")

image = w.Image(format="jpg", width=460)
titre = w.HTML()
listes = {a: w.Dropdown(options=[JE_NE_SAIS_PAS] + VALEURS[a], value=JE_NE_SAIS_PAS,
                        description=QUESTIONS[a], style={"description_width": "330px"},
                        layout=w.Layout(width="620px"))
          for a in ATTRIBUTS}
bouton = w.Button(description="Enregistrer et photo suivante", button_style="success",
                  layout=w.Layout(width="300px"))
etat = {"i": 0}

def afficher():
    if etat["i"] >= len(a_faire):
        titre.value = "<h3>✅ Terminé ! Toutes les photos sont décrites.</h3>"
        image.layout.display = "none"
        bouton.disabled = True
        return
    nom = a_faire[etat["i"]]
    titre.value = f"<h3>{nom} — {len(deja) + etat['i'] + 1} / {len(photos)}</h3>"
    image.value = (PHOTOS_ANONYMES / nom).read_bytes()
    for liste in listes.values():
        liste.value = JE_NE_SAIS_PAS

def enregistrer(_):
    nom = a_faire[etat["i"]]
    nouveau = not DESCRIPTIONS.exists()
    with open(DESCRIPTIONS, "a", newline="", encoding="utf-8") as f:
        ecrivain = csv.writer(f)
        if nouveau:
            ecrivain.writerow(["photo"] + ATTRIBUTS)
        ecrivain.writerow([nom] + [listes[a].value for a in ATTRIBUTS])
    etat["i"] += 1
    afficher()

bouton.on_click(enregistrer)
afficher()
display(w.HBox([image, w.VBox([titre] + list(listes.values()) + [bouton])]))

## C3. Contrôle (sans regarder les espèces)

On compare mon taux de « je ne sais pas » à celui que le générateur suppose. C'est le premier test de réalisme des hypothèses de simulation.

In [ ]:
from src.generate_observations import P_MANQUANT

desc = pd.read_csv(DESCRIPTIONS)
print(len(desc), "photos décrites sur", len(photos))
reponses_moy = (desc[ATTRIBUTS] != JE_NE_SAIS_PAS).sum(axis=1).mean()
print(f"Réponses par photo : {reponses_moy:.1f} sur 12 (générateur : 7,6)")

comparaison_jnsp = pd.DataFrame({
    "moi (test humain)": (desc[ATTRIBUTS] == JE_NE_SAIS_PAS).mean(),
    "générateur (hypothèse)": pd.Series(P_MANQUANT),
}).round(2)
display(comparaison_jnsp)
comparaison_jnsp.plot.barh(figsize=(7, 5), title="Taux de « je ne sais pas » par question");

---
# A. Le modèle

## A1. Générer les observations simulées (compétence C2)

`data/bloc03/profils_especes.csv` = l'export de mes **14 fiches espèces** (10 espèces, mâle et femelle séparés quand ils diffèrent), toutes sourcées (oiseaux.net, quelestcetanimal.com).

Le générateur (`src/generate_observations.py`) fabrique **300 observations par espèce**, comme le ferait un vrai utilisateur :

| Hypothèse | Valeur | Pourquoi |
|---|---|---|
| Erreur de perception (valeur voisine : noir ↔ gris, un cran de taille…) | 10 % | on se trompe surtout sur des choses proches |
| Erreur franche (valeur au hasard) | 3 % | il arrive de se tromper complètement |
| « Je ne sais pas » | 20 % (couleur du dos) à 55 % (œil) | les détails fins se voient moins |
| Réponses minimum par observation | 2 | une description vide n'a pas de sens |

Ces taux sont des **hypothèses** : la partie C3 a commencé à les vérifier.

In [ ]:
!python -m src.generate_observations --per-species 300 --seed 42

from src.generate_observations import lire_fiches
fiches = lire_fiches("data/bloc03/profils_especes.csv")
obs = pd.read_csv("data/bloc03/observations_simulees.csv")
obs.to_csv(BLOC03 / "observations_simulees.csv", index=False)   # copie sur le Drive
obs.head(10)

### Exploration : à quoi ressemblent ces données ?

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
obs[ATTRIBUTS].notna().sum(axis=1).value_counts().sort_index().plot.bar(
    ax=axes[0], title="Nombre de réponses par observation", xlabel="réponses (sur 12)", ylabel="observations")
obs["nom_fr"].value_counts().plot.barh(ax=axes[1], title="Observations par espèce (classes équilibrées)")
plt.tight_layout()

# Exemple : la couleur du dos selon l'espèce (on voit le bruit autour de la vraie valeur)
pd.crosstab(obs["nom_fr"], obs["couleur_dessus"].fillna("(je ne sais pas)"), normalize="index").round(2)

## A2. Prétraitement (compétence C2)

Les réponses sont du **texte** ; un modèle a besoin de **nombres**. Chaque algorithme reçoit le prétraitement qui lui convient, rangé **dans le même Pipeline** que le modèle (`src/chat_model.py`) : l'API reçoit le texte brut et le Pipeline fait le reste.

| Algorithme | Prétraitement | « Je ne sais pas » devient… |
|---|---|---|
| Naive Bayes catégoriel | 1 code entier par réponse (`OrdinalEncoder`) | une catégorie « inconnu » |
| Régression logistique, Random Forest | 1 colonne 0/1 par réponse possible (`OneHotEncoder`) : 53 colonnes | une ligne de zéros (le modèle n'en tire rien) |
| HistGradientBoosting | 1 code entier par réponse, attributs déclarés catégoriels sauf taille, longueur du bec et queue (ordonnés) | `NaN`, géré nativement |

Pourquoi le one-hot pour la régression logistique ? Coder « noir = 0, gris = 1, brun = 2 » ferait croire au modèle que brun est « plus » que noir : il n'y a pas d'ordre entre des couleurs.

In [ ]:
from src.chat_model import preparer_X, construire_modeles

X = preparer_X(obs)
y = obs["espece_id"]
CLASSES = sorted(y.unique())
NOMS = dict(zip(fiches["espece_id"], fiches["nom_fr"]))

# Ce que voit le modèle de régression logistique après prétraitement
prep = construire_modeles()["Régression logistique"].named_steps["prep"].fit(X)
print("Avant :", X.shape, "-> après one-hot :", prep.transform(X).shape)
X.head(3)

## A3. Comparer les 5 algorithmes (compétences C1 et C3)

**Méthode** : validation croisée stratifiée en **5 plis** sur les 3 000 observations simulées → chaque indicateur en **moyenne ± écart-type**. Même découpage pour tous (graine 42). Tout est tracé dans **MLflow**.

**Indicateurs de succès (définis à l'avance)** :
- **log-loss** (entropie croisée) = la **loss optimisée** par l'entraînement. Elle punit un modèle qui se trompe *avec assurance* : essentiel, car l'appli affiche un % de confiance. **C'est le critère de choix.**
- **top-3** : la bonne espèce est dans les 3 proposées (ce que voit l'utilisateur).
- **accuracy** (top-1) et **macro-F1** (moyenne des F1 par espèce : aucune espèce n'est sacrifiée).

| Algorithme | Famille | Pourquoi le tester |
|---|---|---|
| Référence (proportions) | aucune | point de départ : 10 % si le modèle n'apprend rien |
| Naive Bayes catégoriel | probabiliste | fait pour des attributs catégoriels, très rapide, probabilités naturelles |
| Régression logistique | linéaire | simple, explicable (un poids par réponse), probabilités bien calibrées |
| Random Forest | ensemble d'arbres (bagging) | capte les combinaisons de réponses |
| HistGradientBoosting | ensemble d'arbres (boosting) | souvent le plus fort sur données tabulaires, gère les trous |

In [ ]:
import mlflow
from sklearn.model_selection import StratifiedKFold, cross_validate
from src.chat_model import scorers

mlflow.set_tracking_uri(f"file:{DRIVE}/mlruns_bloc03")
mlflow.set_experiment("bloc03_mode_chat")

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lignes = []
for nom, modele in construire_modeles().items():
    r = cross_validate(modele, X, y, cv=CV, scoring=scorers(CLASSES))
    ligne = {
        "algorithme": nom,
        "log_loss": -r["test_log_loss"].mean(), "log_loss_std": r["test_log_loss"].std(),
        "top3": r["test_top3"].mean(),
        "accuracy": r["test_accuracy"].mean(), "accuracy_std": r["test_accuracy"].std(),
        "f1_macro": r["test_f1_macro"].mean(),
        "temps_entrainement_s": r["fit_time"].mean(),
    }
    lignes.append(ligne)
    with mlflow.start_run(run_name=f"cv5 - {nom}"):
        mlflow.log_params({"algorithme": nom, "validation": "StratifiedKFold 5 plis", "graine": 42,
                           "observations": len(X)})
        mlflow.log_metrics({k: v for k, v in ligne.items() if k != "algorithme"})

comparaison = pd.DataFrame(lignes).set_index("algorithme").sort_values("log_loss")
comparaison.to_csv(BLOC03 / "comparaison_algorithmes.csv")
comparaison.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
c = comparaison.drop(index="Référence (proportions)")
c["log_loss"].plot.barh(ax=axes[0], xerr=c["log_loss_std"], title="Log-loss (plus bas = mieux)")
c["accuracy"].plot.barh(ax=axes[1], xerr=c["accuracy_std"], title="Accuracy (plus haut = mieux)", xlim=(0.8, 1))
plt.tight_layout()

### Lecture des résultats

*(À compléter avec mes chiffres.)* Lors d'un essai préparatoire, l'ordre était : Naive Bayes ≈ 0,95 d'accuracy et la meilleure log-loss, puis régression logistique, Random Forest, HistGradientBoosting, tous au-dessus de 0,93 ; la référence à 0,10.

⚠️ **Point honnête à dire au jury** : mon générateur tire chaque attribut **indépendamment** des autres, une fois l'espèce connue. C'est **exactement l'hypothèse du Naive Bayes** : sur ces données, il est avantagé par construction. Un vrai observateur fait des erreurs liées (mauvaise lumière → toutes les couleurs fausses). **C'est le test humain (A5) qui départage pour de vrai.**

## A4. Réglage des 2 meilleurs (hyperparamètres)

On règle les 2 algorithmes en tête (hors référence) avec une recherche en grille, **toujours en validation croisée et sur la log-loss**. Le test humain n'est pas utilisé ici.

In [ ]:
from sklearn.model_selection import GridSearchCV

GRILLES = {
    "Naive Bayes catégoriel": {"modele__alpha": [0.1, 0.5, 1.0, 2.0, 5.0]},          # lissage
    "Régression logistique": {"modele__C": [0.1, 0.3, 1.0, 3.0, 10.0]},              # régularisation
    "Random Forest": {"modele__n_estimators": [200, 500], "modele__min_samples_leaf": [1, 2, 5]},
    "HistGradientBoosting": {"modele__learning_rate": [0.05, 0.1], "modele__max_depth": [None, 4]},
}
finalistes = [n for n in comparaison.index if n != "Référence (proportions)"][:2]
print("Finalistes :", finalistes)

regles = {}
for nom in finalistes:
    recherche = GridSearchCV(construire_modeles()[nom], GRILLES[nom], cv=CV, scoring="neg_log_loss", n_jobs=-1)
    recherche.fit(X, y)
    regles[nom] = {"modele": recherche.best_estimator_, "params": recherche.best_params_,
                   "log_loss_cv": -recherche.best_score_}
    print(f"{nom} : meilleurs réglages {recherche.best_params_} -> log-loss {-recherche.best_score_:.3f}")
    with mlflow.start_run(run_name=f"réglage - {nom}"):
        mlflow.log_params({"algorithme": nom, **recherche.best_params_})
        mlflow.log_metric("log_loss", -recherche.best_score_)

# Choix fixé AVANT de regarder le test humain : la meilleure log-loss en validation croisée
CHOIX = min(regles, key=lambda n: regles[n]["log_loss_cv"])
print("\n➡️ Modèle retenu :", CHOIX)

## A5. La note honnête : le test humain

Le modèle **retenu en A4** est noté **une seule fois** sur mes descriptions de la partie C. L'autre finaliste est affiché pour information : il ne change pas le choix (sinon ce serait choisir sur le test).

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from src.chat_model import evaluer

if not DESCRIPTIONS.exists():
    print("⏸️ Pas encore de descriptions : fais la partie C, puis relance cette cellule.")
else:
    humain = pd.read_csv(DESCRIPTIONS).merge(pd.read_csv(CORRESPONDANCE)[["photo", "espece_id"]], on="photo")
    X_h, y_h = preparer_X(humain), humain["espece_id"]
    print(len(humain), "descriptions humaines")

    lignes = []
    for nom, r in regles.items():
        m_cv = comparaison.loc[nom]
        m_h = evaluer(r["modele"], X_h, y_h)
        lignes.append({"algorithme": nom + (" ✅ retenu" if nom == CHOIX else ""),
                       "accuracy simulé (CV)": m_cv["accuracy"], "accuracy humain": m_h["accuracy"],
                       "top3 simulé (CV)": m_cv["top3"], "top3 humain": m_h["top3"],
                       "log-loss simulé (CV)": r["log_loss_cv"], "log-loss humain": m_h["log_loss"],
                       "macro-F1 humain": m_h["f1_macro"]})
    test_humain = pd.DataFrame(lignes).set_index("algorithme")
    test_humain.to_csv(BLOC03 / "resultats_test_humain.csv")
    display(test_humain.round(3))

    METRIQUES_HUMAIN = evaluer(regles[CHOIX]["modele"], X_h, y_h)
    with mlflow.start_run(run_name=f"test humain - {CHOIX}"):
        mlflow.log_params({"algorithme": CHOIX, **regles[CHOIX]["params"], "jeu": "descriptions humaines"})
        mlflow.log_metrics(METRIQUES_HUMAIN)

    fig, ax = plt.subplots(figsize=(8, 7))
    ConfusionMatrixDisplay.from_predictions(
        y_h.map(NOMS), pd.Series(regles[CHOIX]["modele"].predict(X_h)).map(NOMS),
        normalize="true", xticks_rotation=60, values_format=".1f", ax=ax, colorbar=False)
    ax.set_title(f"Test humain — {CHOIX} (ligne = espèce réelle)")
    plt.tight_layout()

### Lecture du test humain

*(À compléter.)* Questions à se poser :
- **Écart simulé → humain** : s'il est grand, mes hypothèses de simulation sont trop optimistes (le test humain est plus dur). Ce n'est pas un échec : c'est un **résultat**, et une piste d'amélioration (ex. : recalibrer le générateur sur mes taux réels de « je ne sais pas »).
- **Le top-3** reste-t-il élevé ? C'est ce que voit l'utilisateur.
- **Quelles confusions ?** Les mêmes pièges que pour le modèle photo (oiseaux noirs, gris, colibris) ?

## A6. Quelles questions servent le plus ? (explicabilité)

**Importance par permutation** : on mélange les réponses d'une question et on mesure combien la log-loss se dégrade. Plus elle se dégrade, plus la question est utile.

In [ ]:
from sklearn.inspection import permutation_importance

echantillon = obs.sample(1000, random_state=42)
imp = permutation_importance(regles[CHOIX]["modele"], preparer_X(echantillon), echantillon["espece_id"],
                             scoring="neg_log_loss", n_repeats=5, random_state=42)
importance = pd.Series(imp.importances_mean, index=ATTRIBUTS).sort_values()
importance.plot.barh(figsize=(7, 5), title=f"Importance des questions ({CHOIX})", xlabel="hausse de la log-loss");

## A7. Sauvegarder le modèle pour l'API

Le modèle retenu est **ré-entraîné sur les 3 000 observations**, puis sauvegardé avec tout ce dont l'API a besoin, dans **un seul fichier** : `chat_antilles.joblib` (modèle + prétraitement, questions, réponses possibles, noms des espèces, P(réponse | espèce) pour choisir la question suivante, métriques).

In [ ]:
import sklearn
from sklearn.base import clone
from src.chat_model import sauver_artefact

final = clone(regles[CHOIX]["modele"]).fit(X, y)
metriques = {
    "validation_croisee": comparaison.loc[CHOIX].to_dict(),
    "reglages": regles[CHOIX]["params"],
    "test_humain": globals().get("METRIQUES_HUMAIN"),
}
chemin = BLOC03 / "chat_antilles.joblib"
sauver_artefact(chemin, final, CHOIX, obs, fiches, metriques, region="antilles")
with mlflow.start_run(run_name=f"modèle final - {CHOIX}"):
    mlflow.log_params({"algorithme": CHOIX, **regles[CHOIX]["params"]})
    mlflow.log_artifact(str(chemin))

print("✅ Modèle sauvegardé :", chemin, f"({chemin.stat().st_size / 1024:.0f} Ko)")
print(f"⚠️ Version de scikit-learn à installer sur ton PC : scikit-learn=={sklearn.__version__}")

### Essai du mode chat ici même, avant l'API

On joue le rôle d'un utilisateur qui a vu un **Sucrier à ventre jaune** et répond aux questions que pose le moteur de l'API (`app/chat.py`).

In [ ]:
import joblib
from app.chat import ChatEngine, phrase

moteur = ChatEngine(joblib.load(chemin))
vu = fiches[fiches["espece_id"] == "Coereba_flaveola"].iloc[0]   # ce que « l'utilisateur » a vu
reponses = {}
while True:
    tour = moteur.tour(reponses)
    top3 = ", ".join(f"{e['fr']} {p:.0%}" for e, p in tour.predictions[:3])
    print(f"Réponses : {reponses}\n   -> {top3}\n   -> {phrase(tour)}", tour.question or "")
    if tour.termine:
        break
    reponses[tour.question] = vu[tour.question] if isinstance(vu[tour.question], str) else JE_NE_SAIS_PAS

---
# B. Brancher le modèle sur l'API (sur ton PC, pas dans Colab)

Tout le code est déjà dans la branche `bloc03-mode-chat` (étape ⓪) : `app/chat.py` (le moteur), la route `POST /chat` et `GET /chat/questions` dans `app/main.py`, les tests `tests/test_chat.py`, la collection Bruno `bruno/`.

### B1. Installer les bibliothèques (terminal VS Code, dans le dossier `birdsfinder`)
Tu es déjà sur la branche `bloc03-mode-chat` (étape ⓪).
```
.venv\Scripts\activate
pip install -r requirements.txt
pip install scikit-learn==X.Y.Z
```
👉 Remplace `X.Y.Z` par la version affichée en A7. **C'est important** : un modèle joblib ne se relit correctement qu'avec la même version de scikit-learn.

### B2. Copier le modèle
Depuis le Drive (`birdsfinder/bloc03/chat_antilles.joblib`) vers le dossier `models/` du projet.

### B3. Lancer l'API
```
python -m uvicorn app.main:app --reload
```
Puis ouvrir **http://localhost:8000/docs** : la route `POST /chat` apparaît, avec un bouton « Try it out ».

### B4. La démo Bruno
1. Bruno > **Open Collection** > choisir le dossier `bruno/` du projet.
2. En haut à droite, choisir l'environnement **local**.
3. Lancer les requêtes dans l'ordre :

| Requête | Ce qu'elle montre |
|---|---|
| 01 Health, 02 Regions | l'API tourne |
| 03 Chat - les 12 questions | `GET /chat/questions` |
| 04 → 07 Chat - tours 1 à 4 | la confiance **monte** à chaque réponse (Sucrier : ~20 % → ~95 %) et l'API choisit la question suivante |
| 08 Chat - je ne sais pas | une réponse « je ne sais pas » est acceptée et la question n'est plus reposée |
| 09 Chat - erreur | une réponse impossible → **400** avec la liste des choix |
| 10 Stats | le compteur `chat_requests` |

Le champ `gains` montre, pour chaque question encore possible, combien elle réduirait l'incertitude (en bits) : c'est le **gain d'information**. La `question_suivante` est celle qui a le plus grand gain.

### B5. Les tests
```
pytest -q
```
Les tests du chat utilisent un **faux modèle** (comme ceux de la photo) : ils passent aussi dans la CI GitHub sans scikit-learn.

### B6. MLflow (pour le jury)
Les essais sont dans le Drive, dossier `birdsfinder/mlruns_bloc03`. Sur le PC, avec Google Drive pour ordinateur :
```
mlflow ui --backend-store-uri "G:\Mon Drive\birdsfinder\mlruns_bloc03"
```
(adapter la lettre du lecteur), puis **http://localhost:5000**.

### 🛟 Si ça coince
| Symptôme | Cause probable | Solution |
|---|---|---|
| `/chat` répond **503** « introuvable » | le fichier n'est pas dans `models/` ou mal nommé | vérifier `models/chat_antilles.joblib` |
| `/chat` répond **503** « illisible » | versions de scikit-learn différentes entre Colab et le PC | `pip install scikit-learn==` la version de A7 |
| `ModuleNotFoundError: sklearn` ou `pandas` | requirements pas réinstallés | `pip install -r requirements.txt` |
| La question suivante n'est pas celle de Bruno | normal si un autre algorithme a été retenu | les réponses de Bruno restent valides ; la confiance doit quand même monter |

---
# ✅ Bilan pour le livret

- **Données** : 14 fiches sourcées → 3 000 observations simulées (hypothèses chiffrées) + 80 descriptions humaines anonymisées.
- **Prétraitement (C2)** : encodage adapté à chaque famille, gestion explicite du « je ne sais pas ».
- **Choix de l'algorithme (C1)** : 5 algorithmes de familles différentes, validation croisée 5 plis, critère fixé à l'avance (log-loss), réglage des 2 finalistes, choix fait **avant** le test humain.
- **Entraînement (C3)** : loss = entropie croisée (log-loss), indicateurs top-1, top-3, macro-F1 ; note finale sur données humaines.
- **Intégration** : `POST /chat` avec choix de la question suivante par gain d'information, tests automatisés, démo Bruno.
- **Limites à assumer** : données d'entraînement simulées (indépendance des attributs → avantage au Naive Bayes), test humain fait par la conceptrice, 80 descriptions seulement.